# Lab 03 B Ask the Manual - Retrieval Augmented Generation

In this lab, you will build a RAG assistant that answers questions about the Quillbrew QB-500 manuals. You will chunk and index the manuals, then compare two ways of retrieving a passage before the model writes an answer. In naïve RAG, the customer question is compared with passages in a vector database. The search returns the top N passages closest to the question. The language model then writes an answer from those passages only. It cites the passages it used, and it replies IDK when they do not contain the answer.

Manuals use technical terms. Customers describe the same problem in everyday words. A question such as "my coffee tastes burnt" may not sit close to a passage titled "over-extraction," so the right chunk is easy to miss.

One way to address that gap is to have the model write a few customer-style questions for each chunk, then store those questions as the documents in a second index. A new question is matched to the closest generated questions. Each generated question still points back to its chunk, so retrieval can return the original passages for the answer. This is a hypothesis about the index. It does not guarantee a better result than naïve RAG.


## 1.1 Install the packages you will need

You need a client for the language model, a vector database to search passages, and pandas to compare the two indexes later. Install them once.


In [ ]:
!pip install -q openai chromadb pandas


## 1.2 Connect to the model and name the models you will use

One model writes answers and questions. Another turns text into vectors, so a question can be compared with a passage by meaning rather than by shared keywords. The API key stays in this session and is not saved in the notebook.


In [ ]:
import os, re, json, glob
from getpass import getpass

from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

# >>> API KEY: the only place you enter it. Paste your key into the prompt box
# >>> that appears when you run this cell (it stays hidden and is not saved).
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()   # reads OPENAI_API_KEY automatically
GEN_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"
DATA_DIR = "lab_data"

## 1.3 Load the manuals and the questions you will test

The five manuals are the knowledge. The golden questions are the test: some are answered in one section, some need several sections, and some are not in the manuals. Without questions whose answers you already know, you cannot tell whether retrieval worked.


In [ ]:
manuals = {}
for path in sorted(glob.glob(f"{DATA_DIR}/manuals/*.md")):
    doc_id = os.path.basename(path).replace(".md", "")
    with open(path, encoding="utf-8") as f:
        manuals[doc_id] = f.read()

with open(f"{DATA_DIR}/golden_set.json", encoding="utf-8") as f:
    golden = json.load(f)

print(f"Loaded {len(manuals)} manuals and {len(golden)} golden queries")
for doc_id, text in manuals.items():
    print(f"  {doc_id}: {len(text.split())} words")

## 2.1 Parse data from source

Each manual is one markdown file. Parsing it means taking the heading structure apart, so one section becomes one chunk. A whole manual is too long, and it mixes many topics. A customer asks about one problem, so retrieval needs a passage about that problem. Splitting on each heading keeps the section intact. The document title stays on the chunk, so a passage still makes sense when it is retrieved alone.


In [ ]:
def slugify(heading):
    return re.sub(r"[^a-z0-9]+", "-", heading.lower()).strip("-")

def chunk_by_heading(doc_id, text):
    """One chunk per '## ' section. Each chunk is prefixed with the document title
    so that it still makes sense when it is retrieved on its own."""
    title = text.splitlines()[0].lstrip("# ").strip()
    sections = re.split(r"\n(?=## )", text)
    chunks = []
    for sec in sections[1:]:                      # sections[0] is the title block
        heading = sec.splitlines()[0].lstrip("# ").strip()
        chunks.append({
            "id": f"{doc_id}#{slugify(heading)}",
            "doc": doc_id,
            "text": f"{title}\n{sec.strip()}",
        })
    return chunks

## 2.2 Build the chunk list the rest of the lab will search

Both indexes later search this same list. If each index used a different split, you would not know whether a better answer came from the index or from different passages.


In [ ]:
chunks = [c for doc_id, text in manuals.items() for c in chunk_by_heading(doc_id, text)]
chunk_lookup = {c["id"]: c["text"] for c in chunks}

print(f"{len(chunks)} heading-based chunks")
print(chunks[0]["id"])
print(chunks[0]["text"][:400])


## 3.1 Store the chunks so a customer question can search them

Retrieval does not reread every manual on each question. An embedding turns a chunk into a vector. Similar meaning lands in nearby vectors. The vector store holds those vectors so a new question can be compared with every chunk at once.


In [ ]:
embed_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY", model_name=EMBED_MODEL   # reuses the key from Task 1.2
)
chroma = chromadb.Client()   # in-memory database

vanilla = chroma.get_or_create_collection("manual_chunks", embedding_function=embed_fn)
vanilla.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"doc": c["doc"]} for c in chunks],
)
print("Vanilla collection size:", vanilla.count())

## 3.2 Return the three chunks closest to a customer question

The closest vectors are the passages most like the question. Three is enough when the answer is spread across more than one section, and small enough that unrelated text does not bury it. This is also the budget the later hit@3 score uses: the right passage counts if it appears anywhere in these three.


In [ ]:
def retrieve_vanilla(query, k=3):
    res = vanilla.query(query_texts=[query], n_results=k)
    return res["ids"][0]

print(retrieve_vanilla("My coffee tastes really bitter, almost burnt."))

## 3.3 Grounded answers and IDK

Generation is not allowed to supply product facts from memory. The prompt gives the model only the retrieved passages and asks it to cite the passage ids it used. IDK is the safe reply when those passages do not contain the answer. A confident wrong error code or warranty claim costs more than sending the customer to a person.


In [ ]:
ANSWER_PROMPT = """You are a customer support assistant for the Quillbrew QB-500 espresso machine.
Answer the customer's question using ONLY the manual excerpts below.
- Cite the excerpt IDs you used in square brackets, for example [04_troubleshooting#1-error-codes].
- If the excerpts do not contain the answer, reply with exactly: IDK

Manual excerpts:
{context}

Customer question: {query}"""

def answer(query, chunk_ids):
    context = "\n\n".join(f"[{cid}]\n{chunk_lookup[cid]}" for cid in chunk_ids)
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": ANSWER_PROMPT.format(context=context, query=query)}],
    )
    return resp.choices[0].message.content.strip()

## 3.4 Run your assistant once

Run this before you change the index. One answerable question and one question that should be IDK are enough to see the whole path: customer question, three passages, then an answer that cites those passages or says IDK. If this path fails, a later comparison will not tell you whether a new index helped.


In [ ]:
def show_standard(query):
    ids = retrieve_vanilla(query)
    print("Customer question:", query)
    for cid in ids:
        print(f"\n[{cid}]")
        print(chunk_lookup[cid])
    print("\nAnswer:", answer(query, ids))

show_standard(next(g["query"] for g in golden if g["answer"] != "IDK"))
print("\n" + "=" * 60 + "\n")
show_standard(next(g["query"] for g in golden if g["answer"] == "IDK"))


## 4 Try question-based indexing

Standard RAG already answers from the manuals. The next part changes only the retrieval step. It is an experiment, not a step you need in order to understand RAG.

The manuals use technical terms, and customers do not. Question-based indexing tries one idea: write a few customer-style questions for each passage in advance, then search those questions instead of the raw text. This is a hypothesis. It does not guarantee that the right passage will be easier to find.


## 4.1 Customer questions for one passage

A customer says "my coffee tastes burnt." The manual says "over-extraction." A question written in the customer's words can sit closer to a future question than the technical paragraph does. Try the prompt on one passage before you generate questions for every chunk.


In [ ]:
QGEN_PROMPT = """Below is a passage from a coffee machine manual.
Write 3 different questions that a customer could ask and that this passage answers.
Write them the way a real customer would, in everyday language, and avoid the manual's
technical terms where a customer would not know them.
Return JSON: {{"questions": ["...", "...", "..."]}}

Passage:
{passage}"""

def generate_questions(chunk_text):
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        temperature=0.7,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": QGEN_PROMPT.format(passage=chunk_text)}],
    )
    return json.loads(resp.choices[0].message.content)["questions"]

print(generate_questions(chunk_lookup["04_troubleshooting#2-over-extraction"]))

## 4.2 Generate those questions for every chunk

One passage only tests the prompt. The new index needs questions for every chunk. A topic with no generated question can never be found by this route. If generation is too slow, load the saved questions instead of calling the model again.


In [ ]:
CHECKPOINT = f"{DATA_DIR}/generated_questions.json"
USE_CHECKPOINT = False   # set to True if generation fails or is too slow

if USE_CHECKPOINT:
    with open(CHECKPOINT, encoding="utf-8") as f:
        generated = json.load(f)
else:
    generated = {c["id"]: generate_questions(c["text"]) for c in chunks}

print(f"{sum(len(v) for v in generated.values())} questions for {len(generated)} chunks")

## 4.3 Questions stored with their source chunk

This index stores the generated questions, not the manual paragraphs. Search will compare a customer question with those questions. Each question still has to point back to its passage, because the model needs the original paragraph in order to answer.


In [ ]:
question_ids, question_texts, question_to_chunk = [], [], {}
for chunk_id, qs in generated.items():
    for i, qtext in enumerate(qs):
        qid = f"{chunk_id}::q{i}"           # Chroma id encodes the source chunk
        question_ids.append(qid)
        question_texts.append(qtext)
        question_to_chunk[qid] = chunk_id

qindex = chroma.get_or_create_collection("generated_questions", embedding_function=embed_fn)
qindex.add(
    ids=question_ids,
    documents=question_texts,
    metadatas=[{"chunk_id": question_to_chunk[qid]} for qid in question_ids],
)
print("Question collection size:", qindex.count())

## 4.4 From a matched question back to the passage

The customer question is compared with the indexed questions. Each hit points to a passage. Several questions can point to the same passage, so keep the first hit for each passage until you have three different passages. The answer function from Task 3.3 does not change.


In [ ]:
def retrieve_qindex_matches(query, k=3):
    """Selected passage ids, plus the indexed questions that were used to select them."""
    res = qindex.query(query_texts=[query], n_results=k * 3)
    chunk_ids, matched = [], []
    for qid, qtext in zip(res["ids"][0], res["documents"][0]):
        cid = question_to_chunk[qid]
        matched.append({"question": qtext, "passage_id": cid})
        if cid not in chunk_ids:            # several questions can point to one chunk
            chunk_ids.append(cid)
        if len(chunk_ids) == k:
            break
    return chunk_ids, matched

def retrieve_qindex(query, k=3):
    chunk_ids, _ = retrieve_qindex_matches(query, k)
    return chunk_ids

q = golden[0]["query"]
print("matched questions:", qindex.query(query_texts=[q], n_results=3)["documents"][0])
print("chunks:", retrieve_qindex(q))

## 5.1 Run one answerable question through both indexes

Standard RAG is already working. Here, the same questions go through both indexes so you can see whether question-based indexing retrieved a better passage. The reference answer is not shown. For question-based RAG, the matched indexed questions are shown above the passages: customer question, similar indexed question, original passage, answer.

## 5.2 Run one unanswerable question through both indexes

An answerable question can hide a model that invents facts. An unanswerable question, such as an error code the manual never lists, is where a confident wrong answer costs more than IDK. The same cell runs the first golden question whose reference answer is IDK.

## 5.3 Decide whether each answer is supported

After each example, decide:

- Is the retrieved evidence sufficient?
- Is the answer supported by the passage text?
- Are the citations correct?
- Should the assistant have replied IDK?


In [ ]:
import html
from IPython.display import display, HTML

def show_workflow(query):
    """Run one customer question through both pipelines and show the evidence beside the answer."""
    vanilla_ids = retrieve_vanilla(query)
    q_ids, matched = retrieve_qindex_matches(query)
    runs = [
        ("Standard RAG", vanilla_ids, None),
        ("Question-based RAG", q_ids, matched),
    ]
    header = "".join(f"<th style='padding:8px'>{html.escape(name)}</th>" for name, _, _ in runs)

    def matched_html(items):
        if not items:
            return "—"
        blocks = []
        for item in items:
            blocks.append(
                "<p style='margin:8px 0'>"
                f"{html.escape(item['question'])}<br>"
                f"<strong>{html.escape(item['passage_id'])}</strong>"
                "</p>"
            )
        return "".join(blocks)

    matched_cells, passages, answers = [], [], []
    for _, ids, matched in runs:
        matched_cells.append(
            f"<td style='vertical-align:top; padding:8px'>{matched_html(matched)}</td>"
        )
        blocks = []
        for cid in ids:
            blocks.append(
                f"<p style='margin:8px 0 4px'><strong>{html.escape(cid)}</strong></p>"
                f"<pre style='white-space:pre-wrap; margin:0'>{html.escape(chunk_lookup[cid])}</pre>"
            )
        passages.append(f"<td style='vertical-align:top; padding:8px'>{''.join(blocks)}</td>")
        ans = answer(query, ids)
        answers.append(
            f"<td style='vertical-align:top; padding:8px'><pre style='white-space:pre-wrap; margin:0'>{html.escape(ans)}</pre></td>"
        )
    display(HTML(
        "<div style='margin:8px 0 16px'>"
        f"<p><strong>Customer question.</strong> {html.escape(query)}</p>"
        "<table style='width:100%; border-collapse:collapse' border='1'>"
        f"<tr><th style='padding:8px'></th>{header}</tr>"
        "<tr><th style='padding:8px; vertical-align:top'>Matched indexed questions</th>"
        f"{''.join(matched_cells)}</tr>"
        f"<tr><th style='padding:8px; vertical-align:top'>Retrieved passages</th>{''.join(passages)}</tr>"
        f"<tr><th style='padding:8px; vertical-align:top'>Model answer</th>{''.join(answers)}</tr>"
        "</table></div>"
    ))

CHECK = """
Check:
- Is the retrieved evidence sufficient?
- Is the answer supported by the passage text?
- Are the citations correct?
- Should the assistant have replied IDK?
"""

print("Task 5.1 — answerable question")
show_workflow(next(g["query"] for g in golden if g["answer"] != "IDK"))
print(CHECK)

print("Task 5.2 — unanswerable question")
show_workflow(next(g["query"] for g in golden if g["answer"] == "IDK"))
print(CHECK)


## 6.1 Add one unanswerable question of your own

The golden set already contains unanswerable questions. Yours checks a hole those questions may have missed. The manuals must not contain the answer. Re-running this cell replaces Q10 instead of adding a second copy.


In [ ]:
q10 = {
    "id": "Q10",
    "type": "unanswerable",
    "query": "",   # write a question the manuals do not answer
    "answer": "IDK",
}
# Re-running this cell replaces Q10 instead of appending another copy.
golden = [g for g in golden if g["id"] != "Q10"] + [q10]


## 6.2 Score both indexes on every test question

Two examples can be luck. Scoring every question shows whether one index is actually better. For an answerable question, check whether the right passage was in the three results and whether the answer matches the reference. For an unanswerable question, the only success is a reply that starts with IDK.


In [ ]:
JUDGE_PROMPT = """Compare a support answer to a reference answer.
Reference: {ref}
Answer: {ans}
Reply with one word: CORRECT if the answer contains the key facts of the reference
and nothing that contradicts it, PARTIAL if it contains some of them, or WRONG otherwise."""

def judge(ref, ans):
    resp = client.chat.completions.create(
        model=GEN_MODEL, temperature=0,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(ref=ref, ans=ans)}],
    )
    return resp.choices[0].message.content.strip().upper().rstrip(".")

def evaluate(name, retrieve_fn, k=3):
    rows = []
    for g in golden:
        ids = retrieve_fn(g["query"], k)
        ans = answer(g["query"], ids)
        said_idk = ans.strip().upper().startswith("IDK")
        if g["answer"] == "IDK":
            hit, verdict = None, "CORRECT" if said_idk else "WRONG"
        else:
            hit = any(s in ids for s in g["source"])
            verdict = "WRONG" if said_idk else judge(g["answer"], ans)
        rows.append({"pipeline": name, "id": g["id"], "type": g["type"],
                     "hit@k": hit, "verdict": verdict, "retrieved": ids, "answer": ans})
    return rows

results = evaluate("vanilla", retrieve_vanilla) + evaluate("qindex", retrieve_qindex)

## 6.3 Compare which index found the passage and which answer was right

One accuracy number hides where the pipeline failed. Hit@3 says whether the right passage was retrieved. Correct answers says whether the reply matched the reference. IDK accuracy says whether unanswerable questions were refused. The answering model is the same, so a difference here comes from the index.


In [ ]:
import pandas as pd

df = pd.DataFrame(results).drop_duplicates(["pipeline", "id", "type"], keep="last")
display(df[["pipeline", "id", "type", "hit@k", "verdict"]]
        .pivot(index=["id", "type"], columns="pipeline", values=["hit@k", "verdict"]))

answerable = df[df["type"] != "unanswerable"]
summary = pd.DataFrame({
    "hit@3 (answerable)": answerable.groupby("pipeline")["hit@k"].mean(),
    "correct answers": df.groupby("pipeline")["verdict"].apply(lambda v: (v == "CORRECT").mean()),
    "IDK accuracy (unanswerable)": df[df["type"] == "unanswerable"]
        .groupby("pipeline")["verdict"].apply(lambda v: (v == "CORRECT").mean()),
})
display(summary.round(2))

In [ ]:
for _, row in df[df["type"] == "unanswerable"].iterrows():
    print(f'{row["pipeline"]} | {row["id"]}')
    print("Answer:", row["answer"])
    print()

## 6.4 Pick a wrong answer and explain what went wrong

An average does not say why a question failed. Read the retrieved passage ids and the text the model produced. The failure is retrieval when the right passage is missing, and generation when the right passage was retrieved but the answer still does not follow it.


In [ ]:
failures = df[df["verdict"] != "CORRECT"]
for _, r in failures.iterrows():
    print(f"{r['pipeline']} | {r['id']} | hit@k={r['hit@k']} | {r['verdict']}")
    print("  retrieved:", r["retrieved"])
    print("  answer:", r["answer"][:300], "\n")